# XGBoost Per-Language Models

Bag-of-Words + XGBoost with basic lowercase normalization. Trains one model per
competition language (`lang_abv`) so each vectorizer and classifier fit only that
language's vocabulary and label distribution.

In [ ]:
import json
from pathlib import Path

import joblib
import mlflow
import pandas as pd
from scipy.sparse import hstack
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

## Load Data

Prefer DVC-versioned processed splits. If this notebook is copied into Kaggle, fallback paths can be used.

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
KAGGLE_INPUT_DIR = Path('/kaggle/input/competitions/contradictory-my-dear-watson')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
elif (KAGGLE_INPUT_DIR / 'train.csv').exists():
    full_train_df = pd.read_csv(KAGGLE_INPUT_DIR / 'train.csv')
    test_df = pd.read_csv(KAGGLE_INPUT_DIR / 'test.csv')
    sample_submission = pd.read_csv(KAGGLE_INPUT_DIR / 'sample_submission.csv')
    train_df, val_df = train_test_split(
        full_train_df,
        test_size=0.2,
        random_state=42,
        stratify=full_train_df['label'],
    )
    data_source = 'kaggle_input_fallback_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits or Kaggle input files.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Data loaded. Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, source: {data_source}')

## Text Normalization

In [ ]:
def normalize_basic(value):
    return ' '.join(str(value).lower().split())

## Feature Builder

In [ ]:
def fit_vectorizer(train_frame, ngram_range, max_features):
    vectorizer = CountVectorizer(
        lowercase=True,
        ngram_range=ngram_range,
        max_features=max_features,
    )
    premise_text = train_frame['premise'].fillna('').map(normalize_basic)
    hypothesis_text = train_frame['hypothesis'].fillna('').map(normalize_basic)
    vectorizer.fit(pd.concat([premise_text, hypothesis_text]))
    return vectorizer


def make_features(frame, vectorizer, use_overlap, use_difference):
    premise_text = frame['premise'].fillna('').map(normalize_basic)
    hypothesis_text = frame['hypothesis'].fillna('').map(normalize_basic)

    premise_vec = vectorizer.transform(premise_text)
    hypothesis_vec = vectorizer.transform(hypothesis_text)

    blocks = [premise_vec, hypothesis_vec]
    if use_overlap:
        blocks.append(premise_vec.multiply(hypothesis_vec))
    if use_difference:
        blocks.append(abs(premise_vec - hypothesis_vec))

    return hstack(blocks, format='csr')


def train_per_language_models(train_frame, languages, params):
    models_by_language = {}

    for language in languages:
        language_train = train_frame[train_frame['lang_abv'] == language].reset_index(drop=True)
        vectorizer = fit_vectorizer(
            language_train,
            ngram_range=params['ngram_range'],
            max_features=params['max_features'],
        )
        x_train = make_features(
            language_train,
            vectorizer,
            params['use_overlap'],
            params['use_difference'],
        )
        model = XGBClassifier(
            random_state=42,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            eval_metric='mlogloss',
        )
        model.fit(x_train, language_train['label'].astype(int))
        models_by_language[language] = {
            'model': model,
            'vectorizer': vectorizer,
            'train_rows': len(language_train),
        }
        print(f'  trained {language}: {len(language_train)} rows')

    return models_by_language


def predict_per_language(frame, models_by_language, params):
    predictions = np.empty(len(frame), dtype=int)

    for language, language_frame in frame.groupby('lang_abv', sort=False):
        if language not in models_by_language:
            raise KeyError(f'No trained model for language={language!r}')

        language_indices = language_frame.index.to_numpy()
        x_features = make_features(
            language_frame,
            models_by_language[language]['vectorizer'],
            params['use_overlap'],
            params['use_difference'],
        )
        predictions[language_indices] = models_by_language[language]['model'].predict(x_features)

    return predictions

## Hyperparameters

In [ ]:
RUN_NAME = 'xgboost_basic_per_language'

EXPERIMENT_PARAMS = {
    'run_name': RUN_NAME,
    'norm_type': 'basic',
    'ngram_range': (1, 1),
    'max_features': 20_000,
    'use_overlap': True,
    'use_difference': True,
    'n_estimators': 100,
    'learning_rate': 0.1,
}

LANGUAGES = sorted(train_df['lang_abv'].dropna().unique().tolist())
print(f'Training {len(LANGUAGES)} language-specific models: {LANGUAGES}')

## Train, Evaluate, and Log

In [ ]:
params = EXPERIMENT_PARAMS
run_name = params['run_name']
print(f'Running {run_name} ({len(LANGUAGES)} language models, normalization: {params["norm_type"]})...')

models_by_language = train_per_language_models(train_df, LANGUAGES, params)

val_preds = predict_per_language(val_df, models_by_language, params)
metrics = compute_all_metrics(val_df, val_preds)
test_preds = predict_per_language(test_df, models_by_language, params)

submission = build_submission(sample_submission, test_preds)
submission_path = Path('../submissions') / f'{run_name}.csv'
save_submission(submission, submission_path)

models_path = Path('../submissions') / f'{run_name}_models_by_language.joblib'
languages_path = Path('../submissions') / f'{run_name}_languages.json'

joblib.dump(models_by_language, models_path)
languages_path.write_text(json.dumps(LANGUAGES, indent=2), encoding='utf-8')

serving_type = 'sklearn_per_language_engineered_text_pair_features'
metadata = {
    'run_name': run_name,
    'data_source': data_source,
    'serving_type': serving_type,
    'label_mapping': {0: 'entailment', 1: 'neutral', 2: 'contradiction'},
    'preprocessing': params['norm_type'],
    'languages': LANGUAGES,
    'feature_order': ['premise_vec', 'hypothesis_vec']
    + (['overlap_vec'] if params['use_overlap'] else [])
    + (['difference_vec'] if params['use_difference'] else []),
    'train_rows_by_language': {
        language: bundle['train_rows'] for language, bundle in models_by_language.items()
    },
    'params': {**params, 'ngram_range': list(params['ngram_range'])},
}

mlflow_params = {
    **params,
    'ngram_range': str(params['ngram_range']),
    'model': 'XGBClassifier',
    'random_state': 42,
    'data_source': data_source,
    'serving_type': serving_type,
    'language_count': len(LANGUAGES),
}

with start_notebook_run(
    run_name,
    tags={
        'model_family': 'xgboost',
        'features': 'bow_per_language',
        'serving_type': serving_type,
    },
):
    mlflow.log_params(mlflow_params)
    log_metrics(metrics)
    log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
    mlflow.log_artifact(str(submission_path), artifact_path='submissions')
    log_model_artifacts(
        artifacts={
            'models_by_language.joblib': models_path,
            'languages.json': languages_path,
        },
        metadata=metadata,
        artifact_path='model',
    )

summary = {
    'run_name': run_name,
    'normalization': params['norm_type'],
    'language_count': len(LANGUAGES),
    'accuracy': metrics['accuracy'],
    'f1_macro': metrics['f1_macro'],
    'precision_macro': metrics['precision_macro'],
    'recall_macro': metrics['recall_macro'],
    'submission_path': str(submission_path),
    'serving_type': serving_type,
}

per_language_rows = [
    {
        'language': language,
        'val_support': metrics['per_language'][language]['support'],
        'val_accuracy': metrics['per_language'][language]['accuracy'],
        'val_f1_macro': metrics['per_language'][language]['f1_macro'],
        'train_rows': models_by_language[language]['train_rows'],
    }
    for language in LANGUAGES
]

print('Overall validation metrics:')
pd.Series(summary)
print('Per-language validation metrics:')
pd.DataFrame(per_language_rows).sort_values('val_f1_macro', ascending=False)